In [1]:
import openmc
import numpy as np
import os
from _Geometry_methods import Geometry_helper as gh
date = '300926'


In [20]:
number_of_files = 14
fuel_name = "Fuel_CA_2"

materials = gh.create_materials(fuel_name)
fuel_material, heavy_water, water, blanket_CA, air, mat = gh.set_mat(fuel_name)

print(fuel_material)

def shell_vol_calc(geom_or,geom_ir):
    volume=4/3*np.pi*(geom_or**3-geom_ir**3)
    return volume

redefining materials
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
Material
	ID             =	15
	Name           =	Fuel_CA_2
	Temperature    =	899.9499999999999
	Density        =	4.905 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	True
	S(a,b) Tables  
	Nuclides       
	Li6            =	2.5978647687002928e-06 [ao]
	Li7            =	0.2597838790035587 [ao]
	F19            =	0.6441281138790036 [ao]
	U234           =	4.302802148858144e-05 [ao]
	U235           =	0.004813978945337904 [ao]
	U238           =	0.09120635195004177 [ao]
	U236           =	2.205033580077214e-05 [ao]



In [21]:
def runOpenmc(settings_eigen,geom, mat):
    os.system('rm s*h5')
    os.system('rm p*h5')
    os.system('rm tallies.xml')

    # Reset OpenMC internal IDs to avoid duplicates
    openmc.reset_auto_ids()
    #mat = openmc.Materials.from_xml("materials.xml")
    #tallies = openmc.Tallies.from_xml('tallies.xml')
    eigen_model = openmc.Model(geometry=geom, materials=mat, settings=settings_eigen, tallies = None)
    eigen_model.run()
    sp = openmc.StatePoint(f'statepoint.{settings_eigen.batches}.h5')
    return sp.keff

In [22]:
def runOpenmc_varible_r_shell_model(inner_fuel_rads, outer_fuel_rads, outer_moderater_rads,
                                    with_blanket=False,blanket_ors=None, inner_moderator_heights=None, materials=None,
                                   fuel_name=None):
    nr_of_fuel_atoms = []
    nr_of_moderator_atoms = []
    keffs = []
    for inner_fuel_rad,outer_fuel_rad,outer_moderater_rad,blanket_or,inner_moderator_height in\
    zip(inner_fuel_rads,outer_fuel_rads,outer_moderater_rads,blanket_ors,inner_moderator_heights):
        print(f"inner_fuel_r {inner_fuel_rad:.3g}, outer_fuel_rad {outer_fuel_rad:.3g}",
        f"and outer_moderator_r: {outer_moderater_rad:.3g}, blanket_or: {blanket_or:.3g}",
        f"and inner_moderator_height: {inner_moderator_height:.3g}")
        #Setting new geometry
        geom, mat = gh.set_geom_fuel_shell(inner_fuel_rad,outer_fuel_rad,outer_moderater_rad,
                                           with_blanket=with_blanket,blanket_or=blanket_or, 
                                           inner_moderator_height=inner_moderator_height,
                                           materials=materials,fuel_name=fuel_name)
        
        settings = gh.make_settings(geom)
        #Fixing Li6 at 900 K doesn't have a cross section 
        settings.temperature = {'method': 'nearest','tolerance': 1000}
        
        try:
            #runninng openmc in eigenvalue to find keff of geom
            keff=runOpenmc(settings,geom, mat)
            keffs.append(keff)

            #calc volume and nr of atoms
            fuel_cell = geom.get_cells_by_name('fuel')[0]
            fuel_cell.volume = shell_vol_calc(outer_fuel_rad,inner_fuel_rad)
            nr_of_fuel_atoms.append(fuel_cell.atoms['U235'])

            moderator_cell = geom.get_cells_by_name('moderator')[0]
            moderator_cell.volume = shell_vol_calc(inner_fuel_rad,0) +\
                shell_vol_calc(outer_moderater_rad,outer_fuel_rad)
            nr_of_moderator_atoms.append(moderator_cell.atoms['H2'])
            
        except Exception as e:
            print(f"Error for: inner_fuel_r {inner_fuel_rad:.3g}, outer_fuel_rad {outer_fuel_rad:.3g}\
                    and outer_moderator_r: {outer_moderater_rad:.3g}, blanket_or: {blanket_or:.3g}", e)
            break
            
    return keffs, nr_of_fuel_atoms, nr_of_moderator_atoms

In [23]:
#First cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(30,50,nr_of_runs)
outer_fuel_rads      = np.linspace(71,51,nr_of_runs)
outer_moderater_rads = np.linspace(80,80,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(200,200,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderator_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_15.csv
inner_fuel_r 30, outer_fuel_rad 71 and outer_moderator_r: 80, blanket_or: 100 and inner_moderator_height: 200
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2


rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Ba

...

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.09678
        2/1    1.10321
        3/1    1.15600
        4/1    1.09885
        5/1    1.18267
        6/1    1.12867
        7/1    1.11182
        8/1    1.11647
        9/1    1.11848
       10/1    1.06882
       11/1    1.11009

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

In [25]:
#Second cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=0
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'

name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(30,50,nr_of_runs)
outer_fuel_rads      = np.linspace(71,51,nr_of_runs)
outer_moderater_rads = np.linspace(90,90,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(200,200,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderator_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_16.csv
inner_fuel_r 30, outer_fuel_rad 71 and outer_moderator_r: 90, blanket_or: 100 and inner_moderator_height: 200
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ####

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.05439
        2/1    0.82488
        3/1    0.86009
        4/1    0.82817
        5/1    0.85096
        6/1    0.85068
        7/1    0.86299
        8/1    0.82733
        9/1    0.86466
       10/1    0.85499
       11/1    0.86260

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.04578
        2/1    0.82483
        3/1    0.84952
        4/1    0.86705
        5/1    0.90771
        6/1    0.88757
        7/1    0.86197
        8/1    0.86645
        9/1    0.87598
       10/1    0.84565
       11/1    0.87510

...

 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

In [26]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(50,50,nr_of_runs)
outer_fuel_rads      = np.linspace(60,60,nr_of_runs)
outer_moderater_rads = np.linspace(90,90,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(0,100,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_17.csv
inner_fuel_r 50, outer_fuel_rad 60 and outer_moderator_r: 90, blanket_or: 100 and inner_moderator_height: 0
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    0.78853
        2/1    0.73243
        3/1    0.71119
        4/1    0.74519
        5/1    0.77096
        6/1    0.73004
        7/1    0.68710
        8/1    0.69459
        9/1    0.70831
       10/1    0.72127
       11/1    0.71017

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

...

 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.25676
        2/1    1.17696
        3/1    1.14945
        4/1    1.14596
        5/1    1.17391
        6/1   

In [27]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(50,50,nr_of_runs)
outer_fuel_rads      = np.linspace(60,60,nr_of_runs)
outer_moderater_rads = np.linspace(80,80,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(0,100,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_18.csv
inner_fuel_r 50, outer_fuel_rad 60 and outer_moderator_r: 80, blanket_or: 100 and inner_moderator_height: 0
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    0.71239
        2/1    0.60228
        3/1    0.61362
        4/1    0.64578
        5/1    0.60874
        6/1    0.57810
        7/1    0.66352
        8/1    0.66900
        9/1    0.60416
       10/1    0.59113
       11/1    0.61255

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    0.70485
        2/1    0.65474
        3/1    0.60520
        4/1    0.57763
        5/1    0.64449
        6/1    0.66903
        7/1    0.60739
        8/1    0.61040
        9/1    0.69862
       10/1    0.62807
       11/1    0.63347

...

 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

In [28]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(50,50,nr_of_runs)
outer_fuel_rads      = np.linspace(60,60,nr_of_runs)
outer_moderater_rads = np.linspace(70,70,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(0,100,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_19.csv
inner_fuel_r 50, outer_fuel_rad 60 and outer_moderator_r: 70, blanket_or: 100 and inner_moderator_height: 0
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    0.63321
        2/1    0.47665
        3/1    0.45592
        4/1    0.51556
        5/1    0.45447
        6/1    0.52972
        7/1    0.49980
        8/1    0.45966
        9/1    0.48924
       10/1    0.49500
       11/1    0.52565

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

...

 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Ba

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Ba

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Ba

In [29]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(50,50,nr_of_runs)
outer_fuel_rads      = np.linspace(60,60,nr_of_runs)
outer_moderater_rads = np.linspace(70,100,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(110,110,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_20.csv
inner_fuel_r 50, outer_fuel_rad 60 and outer_moderator_r: 70, blanket_or: 100 and inner_moderator_height: 110
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ####

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.09783
        2/1    0.96197
        3/1    0.92524
        4/1    0.94959
        5/1    0.95685
        6/1    0.91500
        7/1    0.94106
        8/1    0.93608
        9/1    0.99364
       10/1    0.98957
       11/1    0.98878

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.09929
        2/1    0.99059
        3/1    0.96199
        4/1    1.01851
        5/1    1.04276
        6/1    1.01145
        7/1    1.03900
        8/1    1.05225
        9/1    0.96407
       10/1    0.99932
       11/1    0.99524

...

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.43061
        2/1    1.28834
        3/1    1.28245
        4/1    1.

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [30]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(45,45,nr_of_runs)
outer_fuel_rads      = np.linspace(60,60,nr_of_runs)
outer_moderater_rads = np.linspace(70,100,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(110,110,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_21.csv
inner_fuel_r 45, outer_fuel_rad 60 and outer_moderator_r: 70, blanket_or: 100 and inner_moderator_height: 110
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ####

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.06685
        2/1    0.96358
        3/1    0.95302
        4/1    0.94384
        5/1    0.88607
        6/1    0.93493
        7/1    0.90339
        8/1    0.93744
        9/1    0.91557
       10/1    0.99126
       11/1    0.94185

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.05298
        2/1    0.91409
        3/1    0.94216
        4/1    1.00762
        5/1    0.93428
        6/1    0.91399
        7/1    0.99177
        8/1    0.88437
        9/1    1.01382
       10/1    0.95203
       11/1    0.94592

...

 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.34205
        2/1    1.18448
        3/1    1.17710
        4/1    1.13283
        5/1    1.17284
        6/1    1.20746
        7/1    1.15101
        8/1    1.25330
        9/1    1.20325
       10/1    1.18379
       11/1    1.16558

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.58344
        2/1    1.40346
        3/1    1.39919

        4/1    1.35038
        5/1    1.41882
        6/1    1.42531
        7/1    1.42969

        8/1    1.32452
        9/1    1.39803
       10/1    1.37286
       11/1    1.345

In [31]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(45,45,nr_of_runs)
outer_fuel_rads      = np.linspace(55,55,nr_of_runs)
outer_moderater_rads = np.linspace(70,100,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(110,110,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_22.csv
inner_fuel_r 45, outer_fuel_rad 55 and outer_moderator_r: 70, blanket_or: 100 and inner_moderator_height: 110
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ####

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.12125
        2/1    1.00579
        3/1    1.02348
        4/1    0.97820
        5/1    0.94149
        6/1    1.00606
        7/1    1.02749
        8/1    0.98487
        9/1    0.93998
       10/1    0.98433
       11/1    1.00168

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

...

 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.39854
        2/1    1.16888
        3/1    1.22671
        4/1    1.15975
        5/1    1.15473
        6/1    1.20756
        7/1    1.22315
        8/1    1.25281
        9/1    1.16865
       10/1    1.28496
       11/1    1.24255

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.44735
        2/1    1.24519
        3/1    1.25186
        4/1    1.29353
        5/1    1.28864
        6/1   

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Ba

In [32]:
#Third cell for running  
#runOpenmc()

#Setting geometry radius arrays
nr_of_runs = 50
number_of_files +=1
#name = f'ideal_fuel_sphere_{date}_{number_of_files}'
name = f'fuel_sphere_with_blanket_{date}_{number_of_files}'
inner_fuel_rads      = np.linspace(40,40,nr_of_runs)
outer_fuel_rads      = np.linspace(50,50,nr_of_runs)
outer_moderater_rads = np.linspace(70,100,nr_of_runs)
outer_blanket_rads   = np.linspace(100,100,nr_of_runs)
inner_moderator_heights = np.linspace(110,110,nr_of_runs)
with_blanket = True

print(f'filename: {name}.csv')
keffs, nr_of_fuel_atoms, nr_of_moderator_atoms = runOpenmc_varible_r_shell_model(inner_fuel_rads, 
                                outer_fuel_rads, outer_moderater_rads,with_blanket=with_blanket,
                                blanket_ors=outer_blanket_rads, 
                                inner_moderator_heights=inner_moderator_heights, materials=mat,
                                fuel_name=fuel_name)

#print(keffs)
#print(nr_of_fuel_atoms)
#print(nr_of_moderatora_atoms)
keff_values = [keff.nominal_value for keff in keffs]
keff_stds = [keff.std_dev for keff in keffs]

data_to_file = np.column_stack((inner_fuel_rads[:len(keff_values)], outer_fuel_rads[:len(keff_values)], 
                                outer_moderater_rads[:len(keff_values)], outer_blanket_rads[:len(keff_values)],
                                inner_moderator_heights[:len(keff_values)], keff_values, keff_stds, 
                                nr_of_fuel_atoms, nr_of_moderator_atoms))
#Saving data to file
np.savetxt(f'data/{name}.csv',data_to_file,delimiter=",",
           header='inner_fuel_rads, outer_fuel_rads, outer_moderater_rads, outer_blanket_rads,inner_moderator_heights,keff_values, \
           keff_stds, nr_of_fuel_atoms, nr_of_moderator_atoms')

filename: fuel_sphere_with_blanket_300926_23.csv
inner_fuel_r 40, outer_fuel_rad 50 and outer_moderator_r: 70, blanket_or: 100 and inner_moderator_height: 110
Materials defined: 
d2o
Air
Blanket_CA
Fuel_CA_2
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ####

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.13489
        2/1    0.97685
        3/1    0.96758
        4/1    0.99144
        5/1    0.97278
        6/1    1.04291
        7/1    1.01854
        8/1    1.01946
        9/1    1.04882
       10/1    1.03492
       11/1    1.02425

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S

...

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE SIMULATION     <====================

  Bat./Gen.      k            Average k
  =========   ========   ====================
        1/1    1.39021
        2/1    1.20134
        3/1    1.26683
        4/1    1.29129
        5/1    1.20700
        6/1    1.18905
        7/1    1.18473
        8/1    1.19371
        9/1    1.23659
       10/1    1.28348
       11/1    1.21781

/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=1.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=2.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=5.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=3.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=4.
  warn(msg, IDWarning)
rm: cannot remove 'p*h5': No such file or directory
rm: cannot remove 'tallies.xml': No such file or directory


 Reading N14 from /root/nndc_hdf5/N14.h5
 Reading N15 from /root/nndc_hdf5/N15.h5
 Reading O17 from /root/nndc_hdf5/O17.h5
 Reading Ar36 from /root/nndc_hdf5/Ar36.h5
 Reading Ar38 from /root/nndc_hdf5/Ar38.h5
 Reading Ar40 from /root/nndc_hdf5/Ar40.h5
 Reading Li6 from /root/nndc_hdf5/Li6.h5
 Reading Li7 from /root/nndc_hdf5/Li7.h5
 Reading F19 from /root/nndc_hdf5/F19.h5
 Reading Th230 from /root/nndc_hdf5/Th230.h5
 Reading Th232 from /root/nndc_hdf5/Th232.h5
 Reading U234 from /root/nndc_hdf5/U234.h5
 Reading U235 from /root/nndc_hdf5/U235.h5
 Reading U238 from /root/nndc_hdf5/U238.h5
 Reading U236 from /root/nndc_hdf5/U236.h5
 Reading c_D_in_D2O from /root/nndc_hdf5/c_D_in_D2O.h5
 Minimum neutron data temperature: 294 K
 Maximum neutron data temperature: 294 K
 Preparing distributed cell instances...
 Reading plot XML file...
 Writing summary.h5 file...
 Maximum neutron transport energy: 20000000 eV for N15
 Initializing source particles...

 ====================>     K EIGENVALUE S